# PyTorch from the Inside: Tensors, Autograd and Precision

The full code of [PyTorch from the Inside: Tensors, Autograd and Precision](https://residual-stream.lvtan71.workers.dev/posts/pytorch-from-the-inside/), run from [its folder](https://github.com/lvtan71/residual-stream-code/tree/main/posts/pytorch-from-the-inside) in the
order the folder's README gives. Each script is followed by the output it
printed for the post, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/pytorch-from-the-inside.ipynb)

**Hardware:** Runs on a CPU; Colab's free runtime is enough. About 1 min end to end.


The post's numbers come from an Apple M4 laptop. Other hardware and library
versions can change the last digits of timings and of some floating-point
results; the conclusions should not change.

In [ ]:
%pip install -q "numpy==2.*" "torch==2.11.*"

# Setup: fetch the companion repo and work in this post's folder.
import os, subprocess, sys
SLUG = "pytorch-from-the-inside"
ROOT = os.path.abspath("..") if os.path.isdir("../posts") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
os.chdir(os.path.join(ROOT, "posts", SLUG))
print("working in", os.getcwd())

## `tensor_views.py`

A tensor's shape, strides and offset; a transpose and a slice as views of the same memory; `view` failing where `reshape` copies.

In [ ]:
!{sys.executable} tensor_views.py

The post printed:

```text
x: shape (3, 4), strides (4, 1), contiguous True
x.T: shape (4, 3), strides (1, 4), contiguous False, same memory True
x[1]: offset 4, strides (1,);  x[:, 1]: offset 1, strides (4,)
after t[0, 1] = 100: x[1, 0] = 100.0
t.contiguous(): strides (3, 1), same memory False
t.view(12) fails: view size is not compatible with input tensor's size and stride (at least one dimension spans across two contiguous subspaces)
t.reshape(12) works (it copies when it must): [0.0, 100.0, 8.0, 1.0]
```

## `check_autograd.py`

`tiny_autograd.py` against PyTorch on the XOR worked example from "Backpropagation, by Hand", then on 200 random networks with a shared weight and a broadcast bias.

In [ ]:
!{sys.executable} check_autograd.py

The post printed:

```text
loss: ours 0.4932, PyTorch 0.4932
dL/dW1: [-0.2726, 0.0, 0.0, 0.0]   largest gap to PyTorch 0.0e+00
dL/db1: [-0.2726, 0.0]   largest gap to PyTorch 0.0e+00
dL/dw2: [-0.1947, 0.0]   largest gap to PyTorch 0.0e+00
dL/db2: [-0.3894]   largest gap to PyTorch 0.0e+00
200 random cases, shared weight and (1, 5) bias: largest gap to PyTorch 1.1e-16
```

## `graph.py`

Walks the graph PyTorch records for a two-layer network, counts its nodes when a layer is applied once and three times, and shows that `no_grad` records nothing.

In [ ]:
!{sys.executable} graph.py

The post printed:

```text
MmBackward0
  ReluBackward0
    MmBackward0
      AccumulateGrad
  AccumulateGrad
layer applied 1x: 3 nodes in the graph
layer applied 3x: 7 nodes in the graph
under no_grad: grad_fn = None
```

## `timing.py`

Matrix-multiply TFLOP/s on the CPU and the GPU in FP32, FP16 and BF16; a GPU multiply timed with and without synchronizing; per-call cost against 200 queued calls.

In [ ]:
!{sys.executable} timing.py

The post printed:

```text
n =  256:  cpu float32   1.36  mps float32   0.12  mps float16   0.14  mps bfloat16  0.13  TFLOP/s
n = 1024:  cpu float32   1.88  mps float32   1.52  mps float16   2.38  mps bfloat16  2.46  TFLOP/s
n = 4096:  cpu float32   1.45  mps float32   2.99  mps float16   3.43  mps bfloat16  3.42  TFLOP/s
4096 matmul on mps, no synchronize: 0.05 ms 'measured', 47.58 ms real
256 matmul on mps: 230 us per call waiting each time, 30 us per call when 200 are queued (1.12 TFLOP/s)
```

## `float_formats.py`

FP32, FP16 and BF16 side by side: range, smallest values, step at 1.0, rounding examples, 10,000 small updates, and loss scaling for FP16 gradients.

In [ ]:
!{sys.executable} float_formats.py

The post printed:

```text
          exponent mantissa    largest  smallest normal  smallest subnormal  step at 1.0
float32          8       23    3.4e+38         1.18e-38             1.4e-45     1.19e-07
float16          5       10   6.55e+04          6.1e-05            5.96e-08     0.000977
bfloat16         8        7   3.39e+38         1.18e-38            9.18e-41      0.00781

    1e-08 stored as  float16: 0            bfloat16: 1.00117e-08
    70000 stored as  float16: inf          bfloat16: 70144
    1.001 stored as  float16: 1.00098      bfloat16: 1
1 + 10,000 steps of 0.0001 in float32  = 2.0002 (should be 2.0000)
1 + 10,000 steps of 0.0001 in bfloat16 = 1.0000 (should be 2.0000)
gradients [1e-08, 2e-07, 0.0001] in float16: [0, 1.79e-07, 0.0001]
scaled by 1024, stored, unscaled: [1e-08, 2e-07, 0.0001]
```

## `router_torch.py`

The four-way router from "Cross-Entropy and the Softmax Gradient" trained with `torch.optim.AdamW`, next to the hand-written AdamW from "Initialization and Optimizers" on the same mini-batches.

In [ ]:
!{sys.executable} router_torch.py

The post printed:

```text
step    1: loss 1.3863, largest gap between PyTorch and optim.py weights 1.7e-18
step   10: loss 1.1757, largest gap between PyTorch and optim.py weights 2.8e-17
step 3000: loss 0.0428, largest gap between PyTorch and optim.py weights 1.1e-15
validation accuracy 0.750
```

## `step_memory.py`

Bytes per parameter kept by an AdamW training step under `torch.autocast` (BF16 on the CPU): weights, gradients, Adam state, and BF16 weight copies saved for backward.

In [ ]:
!{sys.executable} step_memory.py

The post printed:

```text
weights        4.0 bytes/param  (float32)
gradients      4.0 bytes/param  (float32)
Adam m and v   8.0 bytes/param  (float32)
total         16.0 bytes/param for 8,393,728 parameters
during the step: 1 BF16 weight copies saved for backward, 1.0 more bytes/param
```

## `fusion.py`

Two chains of elementwise operations, eager against `torch.compile`, next to the time of one read and one write (`x.clone()`).

In [ ]:
!{sys.executable} fusion.py

The post printed:

```text
GELU, scale, shift, add  eager   9.7 ms   compiled  4.0 ms (2.4x)   first call 1.6 s   same result: True
scale, shift, add        eager   6.7 ms   compiled  1.8 ms (3.7x)   first call 0.0 s   same result: True
one read + one write       1.6 ms  (x.clone(), the floor for any one-pass kernel)
```